In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.font_manager as fm
from matplotlib.legend_handler import HandlerPatch
from matplotlib.path import Path
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import xarray as xr
import pandas as pd
import seaborn as sns
import seaborn_image as isns
import cmasher as cm
import warnings
import string
from scipy.stats import linregress
import numpy as np
import copy

In [6]:
warnings.filterwarnings('ignore')

In [7]:
sns.set_palette("pastel")
isns.set_context(mode="notebook", fontfamily="MS PMincho")
sns.set_context(font_scale=0.8)

In [8]:
font_path = "/Users/chiaraciscato/Desktop/GEOMAR/2024_ciscato_oae_seasonality/MS PMincho.ttf"
fm.fontManager.addfont(font_path) 

In [9]:
def create_discrete_cmap(cmap, n):

    colors = cmap(np.linspace(0, 1, n))
    discrete_cmap = mcolors.ListedColormap(colors)
    bounds = np.linspace(0, 100, n+1)
    norm = mcolors.BoundaryNorm(bounds, discrete_cmap.N)
    
    return discrete_cmap, norm
    
n = 25

In [1]:
def month_x_labels(ax):
    
    month_label = ['J','F','M','A','M','J','J','A','S','O','N','D']
    ax.set_xticks(np.arange(1, 13, 1))
    ax.set_xticklabels(month_label, fontsize=12)

In [1]:
def var_units():
    # variable units
    var_units = {
                        'ALK' : {'Alkalinity' : r'μ mol $\mathregular{kg^{-1}}$'},
                        'DIC' : {'DIC' : r'μ mol $\mathregular{kg^{-1}}$'},
                        'co2flux' : {'$\mathregular{CO_{2}}$ flux' : r'kg $\mathregular{m^{-2} \ y^{-1}}$'},
                        'fco2' : {'Ocean p$\mathregular{CO_{2}}$' : r'μatm'},
                        'ph': {'pH': 'unit'},
                        'somxl010': {'MLD': 'm'},
                        'bgc_diag_pp': {'NPP': r'mmol $\mathregular{m^{-3} \ y^{-1}}$'},
                        'sosstsst': {'SST': r'$^\circ$C'},                   
                }

    return var_units

In [1]:
def lat_lon_labels():
    
    # degree symbol in string
    d = '$^\circ$' 
    # latitude coordinates
    lat_labels = [f'38{d}N', f'44{d}N',f'50{d}N', f'56{d}N', f'62{d}N', f'69{d}N',f'76{d}N'] 
    # longitude coordinates
    lon_labels = [f'28{d}W', f'20{d}W', f'12{d}W',f'4{d}W', f'4{d}E',f'12{d}E']

    return lat_labels, lon_labels

In [2]:
def seasonal_avg(data):
    
    m_length = data.time_counter.dt.days_in_month
    monthly_avg = (data * m_length).groupby("time_counter.month").mean(dim="time_counter") / m_length.groupby("time_counter.month").mean(dim="time_counter")
    
    return monthly_avg